# <u>Group S</u>: Anne Faury, Héloise Lordez, Cyprien Charlaté, Lucas Gorrec, William Roose


# Baby Names — V2 (updated after peer feedback)

This notebook implements the three favourite visualisations from our first week, now revised after peer review. For the design discussion (what changed, why, and whether it improved the result), see our follow-up post on the Moodle forum.

### Reminder:
<u>Visualization 1</u>: 
- How do baby names evolve over time?
- Are there names that have consistently remained popular or unpopular?
- Are there some that have were suddenly or briefly popular or unpopular?
- Are there trends in time?

<u>Visualization 2</u>: 
- Is there a regional effect in the data? 
- Are some names more popular in some regions? 
- Are popular names generally popular across the whole country?

<u>Visualization 3</u>: 
- Are there gender effects in the data? 
- Does popularity of names given to both sexes evolve consistently? (Note: this data set treats sex as binary; this is a simplification that carries into this assignment but does not generally hold.)


> **Data notes**  
> - Rows with `preusuel == "_PRENOMS_RARES"` (rare names) are dropped.  
> - Rows with `dpt == "XX"` (unknown department) and `annais == "XXXX"` (unknown year) are dropped.  
> - `sexe`: 1 = male, 2 = female.

In [ ]:
import altair as alt
import pandas as pd
import geopandas as gpd

# html renderer outputs interactive Vega-Embed HTML (text/html) that VS Code renders natively.
# disable_max_rows inlines all data so no external file references are needed.
alt.data_transformers.disable_max_rows()
alt.renderers.enable('html')

DATA  = "../Names hints/dpt2020.csv"
GEO_M = "../Names hints/departements-avec-outre-mer.geojson"

# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA, sep=";", dtype={"dpt": str, "annais": str})
df = df[
    (df.preusuel != "_PRENOMS_RARES") &
    (df.dpt      != "XX")             &
    (df.annais   != "XXXX")
]
df["annais"] = df["annais"].astype(int)
df["nombre"] = df["nombre"].astype(int)
df["sexe"]   = df["sexe"].astype(int)

print(f"Rows: {len(df):,}  |  Unique names: {df.preusuel.nunique():,}  "
      f"|  Years: {df.annais.min()}–{df.annais.max()}  "
      f"|  Departments: {df.dpt.nunique()}")
df.head(3)

## Visualization 1 — Name popularity over time (V2)

V2 adds a sliding top-20 (decade slider) and a free-text search overlay for any name in the dataset. See the forum post for the full design discussion.

In [ ]:
# ── Data prep ───────────────────────────────────────────────────────────────
national = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()

YEAR_MIN, YEAR_MAX = int(national.annais.min()), int(national.annais.max())
WINDOW = 10  # sliding window size, in years

# Rolling top-20: for every possible 10-year window, find the top-20 names.
# Store, per name, the list of window-start years where it appears in that
# top-20 (a real Python list -> Altair serialises it as a JS array).
pivot_v1 = national.pivot_table(index="preusuel", columns="annais", values="nombre", fill_value=0)
for y in range(YEAR_MIN, YEAR_MAX + 1):
    if y not in pivot_v1.columns:
        pivot_v1[y] = 0
pivot_v1 = pivot_v1[sorted(pivot_v1.columns)]

decade_starts_v1 = list(range(YEAR_MIN, YEAR_MAX - WINDOW + 2))
name_to_decades_v1 = {}
for start in decade_starts_v1:
    window_sum = pivot_v1[list(range(start, start + WINDOW))].sum(axis=1)
    for name in window_sum.nlargest(20).index:
        name_to_decades_v1.setdefault(name, []).append(start)

# Also keep the all-time top-20 so long-term classics are never dropped.
top20_overall_v1 = national.groupby("preusuel")["nombre"].sum().nlargest(20).index.tolist()
for name in top20_overall_v1:
    name_to_decades_v1.setdefault(name, [])

names_needed_v1 = sorted(name_to_decades_v1.keys())
decades_compact_v1 = pd.DataFrame({
    "preusuel":    list(name_to_decades_v1.keys()),
    "decades_arr": [sorted(v) for v in name_to_decades_v1.values()],
})

# Full 1900-2020 trace for every name that is in the top-20 of at least one window.
data_v1 = (
    national[national.preusuel.isin(names_needed_v1)]
    .merge(decades_compact_v1, on="preusuel", how="left")
)

# Free-text search data: ALL names, in a compact one-row-per-name shape
# (years/counts as lists) so the search filter is evaluated once per name
# rather than once per (name, year) row — keeps it responsive at ~15k names.
data_search_v1 = (
    national.sort_values(["preusuel", "annais"])
    .groupby("preusuel")
    .agg(years=("annais", list), counts=("nombre", list))
    .reset_index()
)

print(f"data_v1: {len(data_v1):,} rows | {data_v1.preusuel.nunique()} names (in >=1 rolling top-20)")
print(f"data_search_v1: {len(data_search_v1):,} rows (1/name) | {data_search_v1.preusuel.nunique()} searchable names")

# ── Interactive params ───────────────────────────────────────────────────────
decade_p1 = alt.param(
    name="v1_decade",
    bind=alt.binding_range(
        min=YEAR_MIN, max=YEAR_MAX - WINDOW + 1, step=1,
        name=f"Top-20 window ({WINDOW} years) — start year: ",
    ),
    value=YEAR_MIN,
)
search_p1 = alt.param(
    name="v1_search",
    bind=alt.binding(input="search", placeholder="Type a full first name (e.g. Kamal)…",
                      name="Search a name: "),
    value="",
)
sel = alt.selection_point(fields=["preusuel"], bind="legend")

in_window_v1 = "indexof(datum.decades_arr, v1_decade) >= 0"

# Exact match only (case-insensitive), to avoid showing several names at
# once while the user is still typing.
search_active1 = "length(v1_search) > 0"
search_match1  = "upper(datum.preusuel) == upper(v1_search)"
search_filter1 = f"{search_active1} && {search_match1}"

# X domain extended past YEAR_MAX to leave room for the search label.
X_DOMAIN_V1 = [YEAR_MIN, YEAR_MAX + 6]
x_enc_v1 = alt.X("annais:Q", title="Year",
                  scale=alt.Scale(domain=X_DOMAIN_V1),
                  axis=alt.Axis(format="d", tickCount=13))

# ── Layer 1: rolling top-20 ───────────────────────────────────────────────────
top20_layer_v1 = (
    alt.Chart(data_v1)
    .mark_line()
    .transform_filter(in_window_v1)
    .encode(
        x=x_enc_v1,
        y=alt.Y("nombre:Q", title="Number of births (national)"),
        color=alt.Color("preusuel:N",
                         legend=alt.Legend(title="Click a name to highlight",
                                            symbolStrokeWidth=3)),
        opacity=alt.condition(sel, alt.value(1.0), alt.value(0.30)),
        strokeWidth=alt.condition(sel, alt.value(3.0), alt.value(1.2)),
        tooltip=[
            alt.Tooltip("preusuel:N", title="Name"),
            alt.Tooltip("annais:Q",   title="Year",   format="d"),
            alt.Tooltip("nombre:Q",   title="Births", format=","),
        ],
    )
    .add_params(sel, decade_p1)
)

# ── Layer 2: search result (black, dashed) ────────────────────────────────────
search_line_v1 = (
    alt.Chart(data_search_v1)
    .mark_line(color="black", strokeDash=[6, 3], strokeWidth=3)
    .transform_filter(search_filter1)
    .transform_flatten(["years", "counts"])
    .encode(
        x=alt.X("years:Q", scale=alt.Scale(domain=X_DOMAIN_V1)),
        y=alt.Y("counts:Q"),
        tooltip=[
            alt.Tooltip("preusuel:N", title="Name (search)"),
            alt.Tooltip("years:Q",    title="Year", format="d"),
            alt.Tooltip("counts:Q",   title="Births", format=","),
        ],
    )
    .add_params(search_p1)
)

# ── Layer 3: name label on the search line's last point ──────────────────────
search_label_v1 = (
    alt.Chart(data_search_v1)
    .transform_filter(search_filter1)
    .transform_flatten(["years", "counts"])
    .transform_window(
        rank="rank()",
        sort=[alt.SortField("years", order="descending")],
        groupby=["preusuel"],
    )
    .transform_filter("datum.rank == 1")
    .mark_text(align="left", dx=8, dy=-4, fontWeight="bold", fontSize=12, color="black")
    .encode(
        x=alt.X("years:Q", scale=alt.Scale(domain=X_DOMAIN_V1)),
        y=alt.Y("counts:Q"),
        text="preusuel:N",
    )
)

v1 = (
    alt.layer(top20_layer_v1, search_line_v1, search_label_v1)
    .resolve_scale(color="independent")
    .properties(
        title=alt.TitleParams(
            "National births per year (1900-2020) — Sliding top-20 + free search",
            subtitle="Slider recomputes the top-20 over a 10-year window. Search overlays any single name from the dataset (exact match), in or out of the top-20.",
            subtitleColor="gray",
            subtitleFontSize=11,
        ),
        width=800,
        height=430,
    )
    .configure_view(strokeWidth=0)
)

v1


## Visualization 2 — Regional effects (V2)

V2 adds a free-text search (top 1,000 names) alongside the original top-3 panel, and an auto/fixed colour-scale toggle scoped to the search overlay. See the forum post for the full design discussion.

In [ ]:
import warnings
import math

# ── 1. National top-3 per year ────────────────────────────────────────────────
nat_v2 = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()
nat_v2["rank_id"] = (
    nat_v2.groupby("annais")["nombre"]
    .rank(method="first", ascending=False)
    .astype(int)
)
top3_yr       = nat_v2[nat_v2["rank_id"] <= 3][["preusuel", "annais", "rank_id"]].copy()
top3_names_v2 = top3_yr["preusuel"].unique().tolist()

# ── 2. Department proportions (per 1 000 births), top-3 only ─────────────────
by_dpt = (
    df[df["preusuel"].isin(top3_names_v2)]
    .groupby(["preusuel", "dpt", "annais"], as_index=False)["nombre"].sum()
)
tot_dpt = (
    df.groupby(["dpt", "annais"], as_index=False)["nombre"]
    .sum().rename(columns={"nombre": "total"})
)
by_dpt = by_dpt.merge(tot_dpt, on=["dpt", "annais"])
by_dpt["prop"] = (by_dpt["nombre"] / by_dpt["total"] * 1_000).round(1)
data_long_v2 = (
    top3_yr.merge(by_dpt, on=["preusuel", "annais"], how="left")
    .fillna({"prop": 0.0})
)

# ── 3. Wide pivot: one row per (dept x rank_id), columns p_YYYY ──────────────
def rank_wide(rid):
    sub = data_long_v2[data_long_v2["rank_id"] == rid]
    w = (
        sub.pivot_table(index="dpt", columns="annais", values="prop", aggfunc="first")
        .fillna(0)
    )
    w.columns = [f"p_{int(y)}" for y in w.columns]
    w = w.reset_index()
    w["rank_id"] = rid
    return w

wide_all_v2 = pd.concat([rank_wide(1), rank_wide(2), rank_wide(3)], ignore_index=True)

# Codes Corsica as "20"; the GeoJSON uses "2A"/"2B".
_corse = wide_all_v2[wide_all_v2["dpt"] == "20"].copy()
if len(_corse) > 0:
    _c2a = _corse.copy(); _c2a["dpt"] = "2A"
    _c2b = _corse.copy(); _c2b["dpt"] = "2B"
    wide_all_v2 = pd.concat(
        [wide_all_v2[wide_all_v2["dpt"] != "20"], _c2a, _c2b],
        ignore_index=True,
    )

# ── 4. Merge geometry — reposition overseas territories + latitude fix ──────
from shapely.affinity import translate as _sha_translate, scale as _sha_scale

depts_geo_v2 = gpd.read_file(GEO_M)

_inset_cfg = {
    "971": (-4.0, 40.6, 0.8),
    "972": (-1.5, 40.6, 0.6),
    "973": ( 1.5, 40.4, 0.8),
    "974": ( 6.0, 40.6, 0.8),
    "976": ( 8.8, 40.6, 0.5),
}

def _reposition(gdf, cfg):
    rows = []
    for _, row in gdf.iterrows():
        if row["code"] not in cfg:
            rows.append(row)
            continue
        b = row.geometry.bounds
        orig_size = max(b[2] - b[0], b[3] - b[1])
        tx, ty, tsize = cfg[row["code"]]
        factor = tsize / orig_size
        g = _sha_scale(row.geometry, xfact=factor, yfact=factor, origin="centroid")
        b2 = g.bounds
        g = _sha_translate(g, tx - (b2[0]+b2[2])/2, ty - (b2[1]+b2[3])/2)
        row = row.copy()
        row["geometry"] = g
        rows.append(row)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)

depts_geo_v2 = _reposition(depts_geo_v2, _inset_cfg)

_LAT_CORR = 1.0 / math.cos(math.radians(46.0))
depts_geo_v2["geometry"] = depts_geo_v2.geometry.apply(
    lambda g: _sha_scale(g, xfact=1.0, yfact=_LAT_CORR, origin=(0, 0))
)
depts_geo_v2 = depts_geo_v2.set_geometry("geometry")

geo_v2 = depts_geo_v2.merge(wide_all_v2, how="inner", left_on="code", right_on="dpt")
geo_v2 = geo_v2.drop(columns=["dpt"])

global_max_v2 = float(data_long_v2["prop"].max())
name_src_v2   = top3_yr[["preusuel", "annais", "rank_id"]].copy()
print(f"geo_v2: {len(geo_v2)} rows | top-3 colour domain [0, {global_max_v2:.1f}]")

# ── 5. Free-text search data (top-1000 national names) ───────────────────────
# Capped at 1000 names to keep the payload manageable (full dataset would be
# ~2M rows even sparse). One row per department (never duplicated) with a
# nested {name: {years, props}} object, so the map geometry is never
# repeated across names — only the matching name's series is looked up by
# key in the browser.
TOP_N_SEARCH_V2 = 1000
nat_totals_v2  = df.groupby("preusuel")["nombre"].sum().sort_values(ascending=False)
top1000_names_v2 = nat_totals_v2.head(TOP_N_SEARCH_V2).index.tolist()

by_dpt_search_v2 = (
    df[df["preusuel"].isin(top1000_names_v2)]
    .groupby(["preusuel", "dpt", "annais"], as_index=False)["nombre"].sum()
    .merge(tot_dpt, on=["dpt", "annais"])
)
by_dpt_search_v2["prop"] = (by_dpt_search_v2["nombre"] / by_dpt_search_v2["total"] * 1_000).round(1)

def _build_names_data(group):
    out = {}
    for name, sub in group.groupby("preusuel"):
        out[name] = {"years": sub["annais"].tolist(), "props": sub["prop"].tolist()}
    return out

search_compact_v2 = (
    by_dpt_search_v2.groupby("dpt")
    .apply(_build_names_data, include_groups=False)
    .reset_index(name="names_data")
)

_corse_s = search_compact_v2[search_compact_v2["dpt"] == "20"].copy()
if len(_corse_s) > 0:
    _cs_2a = _corse_s.copy(); _cs_2a["dpt"] = "2A"
    _cs_2b = _corse_s.copy(); _cs_2b["dpt"] = "2B"
    search_compact_v2 = pd.concat(
        [search_compact_v2[search_compact_v2["dpt"] != "20"], _cs_2a, _cs_2b],
        ignore_index=True,
    )

geo_search_v2 = depts_geo_v2.merge(search_compact_v2, how="inner", left_on="code", right_on="dpt")
geo_search_v2 = geo_search_v2.drop(columns=["dpt"])
print(f"geo_search_v2: {len(geo_search_v2)} rows (1 per department) "
      f"| {len(top1000_names_v2)} searchable names")

# Per-name maximum proportion, used for the search overlay's "fixed" scale.
name_max_lookup_v2 = (
    by_dpt_search_v2.groupby("preusuel")["prop"].max().round(1).to_dict()
)

# ── 6. Params ─────────────────────────────────────────────────────────────────
year_p_v2 = alt.param(
    name="v2_year",
    bind=alt.binding_range(min=1900, max=2020, step=1, name="Year: "),
    value=2000,
)
click_sel_v2 = alt.selection_point(
    fields=["rank_id"],
    name="v2_click",
    empty=False,
    value=[{"rank_id": 1}],
)
search_p_v2 = alt.param(
    name="v2_search",
    bind=alt.binding(input="search", placeholder="Type a full first name (top 1 000 most common)…",
                      name="Search any name: "),
    value="",
)
# "auto" rescales to whichever data is on screen; "fixed" pins the scale —
# see the colour-scale definitions below for how this is scoped.
scale_mode_p_v2 = alt.param(
    name="v2_scale_mode",
    bind=alt.binding_radio(options=["auto", "fixed"], name="Colour scale: "),
    value="auto",
)
# Per-name max lookup, exposed as a JS object so it can be indexed by key
# inside an expression (a colour scale's domain can't reference a row datum).
name_max_lookup_p_v2 = alt.param(
    name="v2_name_max_lookup",
    value=name_max_lookup_v2,
)

# ── 7. Layout & colour scales ──────────────────────────────────────────────────
SW, SH = 205, 220
LW, LH = 530, 565

# Mini-maps and the top-3 large map always auto-scale (their three names
# change every year, so a single all-time ceiling would wash out most
# recent decades — see the blog post for why this differs from the search
# overlay). The search overlay keeps the auto/fixed toggle, pinned to that
# specific name's own all-time maximum in "fixed" mode.
_DOMAIN_MAX_EXPR_LARGE = (
    "length(v2_search) == 0 ? null : "
    "(v2_scale_mode != 'fixed' ? null : v2_name_max_lookup[upper(v2_search)])"
)
COLOR_SMALL = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0),
    # All three mini-maps share one resolved colour scale (see small_row_v2
    # below), so declaring a legend here merges into a single legend, not three.
    legend=alt.Legend(title="For 1 000 births"),
)
COLOR_LARGE = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0, domainMax={"expr": _DOMAIN_MAX_EXPR_LARGE}),
    legend=alt.Legend(title="For 1 000 births"),
)
COLOR_SEARCH = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues", domainMin=0, domainMax={"expr": _DOMAIN_MAX_EXPR_LARGE}),
    legend=None,
)

# ── 8. Name labels (top-3 panels) ─────────────────────────────────────────────
_MEDALS = {1: "\U0001f947", 2: "\U0001f948", 3: "\U0001f949"}

def name_label_v2(rid, w):
    selected = f"v2_click.rank_id == {rid}"
    medal = _MEDALS[rid]
    return (
        alt.Chart(name_src_v2)
        .mark_text(align="center", baseline="middle", fontSize=13, fontWeight="bold")
        .encode(
            text="label:N",
            x=alt.value(w / 2),
            y=alt.value(12),
            color=alt.condition(selected, alt.value("#1a6bb5"), alt.value("#aaaaaa")),
        )
        .transform_filter(f"datum.annais == v2_year && datum.rank_id == {rid}")
        .transform_calculate("label", f"'{medal} ' + datum.preusuel")
        .properties(width=w, height=22)
    )

# ── 9. Small maps (top-3 panels) ──────────────────────────────────────────────
def small_map_v2(rid):
    return (
        alt.Chart(geo_v2)
        .mark_geoshape(stroke="white", strokeWidth=0.4, cursor="pointer")
        .encode(
            color=COLOR_SMALL,
            tooltip=[
                alt.Tooltip("nom:N",  title="Department"),
                alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
            ],
        )
        .transform_calculate("prop", "datum['p_' + v2_year]")
        .transform_filter(f"datum.rank_id == {rid}")
        .add_params(click_sel_v2)
        .project(type="identity", reflectY=True)
        .properties(width=SW, height=SH)
    )

# ── 10. Header above large map ────────────────────────────────────────────────
# Search takes priority over the clicked top-3 panel while it has text.
large_label_v2 = (
    alt.Chart(name_src_v2)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(LW / 2), y=alt.value(12))
    .transform_filter("length(v2_search) == 0 && datum.annais == v2_year && datum.rank_id == v2_click.rank_id")
    .properties(width=LW, height=22)
)
search_label_v2 = (
    alt.Chart(pd.DataFrame({"label_text": ["placeholder"]}))
    # Wording is "no exact match" rather than "not in top 1000": the search
    # is accent-sensitive, so a correctly-listed name typed without its
    # accent (e.g. "stephane") would otherwise show a misleading message.
    .transform_calculate(
        label_text="isValid(v2_name_max_lookup[upper(v2_search)]) ? v2_search : 'No exact match — check spelling/accents'"
    )
    .transform_filter("length(v2_search) > 0")
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(
        text="label_text:N",
        x=alt.value(LW / 2),
        y=alt.value(12),
        color=alt.condition(
            "isValid(v2_name_max_lookup[upper(v2_search)])",
            alt.value("black"),
            alt.value("#b3261e"),
        ),
    )
)

# Subtitle clarifying that the "fixed" ceiling is this name's own peak, not
# a shared/arbitrary value (it can be far smaller than the top-3's scale).
search_subtitle_v2 = (
    alt.Chart(pd.DataFrame({"subtitle_text": ["placeholder"]}))
    .transform_calculate(
        subtitle_text=(
            "length(v2_search) == 0 || !isValid(v2_name_max_lookup[upper(v2_search)]) ? '' : "
            "(v2_scale_mode != 'fixed' ? 'Scale: auto-adjusted to this year’s range' : "
            "'Scale: 0–' + format(v2_name_max_lookup[upper(v2_search)], '.0f') + '‰ — this name’s own all-time peak')"
        )
    )
    .transform_filter("length(v2_search) > 0")
    .mark_text(align="center", baseline="middle", fontSize=10, color="gray", fontStyle="italic")
    .encode(text="subtitle_text:N", x=alt.value(LW / 2), y=alt.value(28))
)

# ── 11. Large map — switches between top-3 selection and free search ─────────
# Both layers share one projection (declared on the combined layer below)
# so their auto-fit doesn't diverge between the two.
large_map_top3_v2 = (
    alt.Chart(geo_v2)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=COLOR_LARGE,
        tooltip=[
            alt.Tooltip("nom:N",  title="Department"),
            alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
        ],
    )
    .transform_calculate("prop", "datum['p_' + v2_year]")
    .transform_filter("length(v2_search) == 0 && datum.rank_id == v2_click.rank_id")
)

large_map_search_v2 = (
    alt.Chart(geo_search_v2)
    # Names are stored uppercase; upper() makes the search case-insensitive.
    .transform_calculate(name_entry="datum.names_data[upper(v2_search)]")
    .transform_filter("length(v2_search) > 0 && isValid(datum.name_entry)")
    .transform_calculate(idx="indexof(datum.name_entry.years, v2_year)")
    # A missing year (idx == -1, sparse storage) means zero births that
    # year, not "name absent" — map it to 0 rather than dropping the row,
    # so it renders as pale blue (in-scale) instead of blank (out-of-scale).
    .transform_calculate("prop", "datum.idx >= 0 ? datum.name_entry.props[datum.idx] : 0")
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=COLOR_SEARCH,
        tooltip=[
            alt.Tooltip("nom:N",  title="Department"),
            alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
        ],
    )
)

large_map_v2 = (
    alt.layer(large_map_top3_v2, large_map_search_v2)
    .resolve_scale(color="shared")
    .add_params(year_p_v2, search_p_v2, scale_mode_p_v2, name_max_lookup_p_v2)
    .project(type="identity", reflectY=True)
    .properties(
        width=LW, height=LH,
        title=alt.TitleParams(
            text=alt.ExprRef("toString(v2_year)"),
            orient="bottom",
            anchor="end",
            fontSize=36,
            fontWeight="bold",
            color="#999999",
            offset=4,
            dx=130,
        ),
    )
)

# ── 12. Assemble ──────────────────────────────────────────────────────────────
labels_row_v2 = alt.hconcat(
    name_label_v2(1, SW), name_label_v2(2, SW), name_label_v2(3, SW), spacing=2
)
# Shared scale across the three mini-maps so the real rank-1/2/3 popularity
# gap is visible as a real colour gap, not just panel order.
small_row_v2 = alt.hconcat(
    small_map_v2(1), small_map_v2(2), small_map_v2(3), spacing=10
).resolve_scale(color="shared")

separator_v2 = (
    alt.Chart(pd.DataFrame({"y": [0]}))
    .mark_rule(color="#222222", strokeWidth=1.5)
    .encode(y=alt.value(1))
    .properties(width=SW * 3 + 15, height=4)
)

note_v2 = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_text(align="center", baseline="middle", fontSize=11, color="gray", fontStyle="italic")
    .encode(text=alt.value("Note: click a mini map to view it large, or type a name to search (top 1 000 names; clear the box to go back to the top-3)"),
            x=alt.value(LW / 2), y=alt.value(9))
    .properties(width=LW, height=18)
)

large_header_row_v2 = (
    alt.layer(large_label_v2, search_label_v2, search_subtitle_v2)
    .properties(width=LW, height=34)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    v2_new = (
        alt.vconcat(
            labels_row_v2,
            small_row_v2,
            separator_v2,
            note_v2,
            large_header_row_v2,
            large_map_v2,
            spacing=6,
        )
        .resolve_scale(color="independent")
        .configure_view(strokeWidth=0)
    )

v2_new


## Visualization 3 — Gender effects on unisex names (V2)

V2 adds a raw/percent toggle and a mini area chart of total popularity above the butterfly chart. See the forum post for the full design discussion.

In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
nat_sex_v3 = df.groupby(["preusuel", "annais", "sexe"], as_index=False)["nombre"].sum()

# Unisex names: at least 200 total births for each sex
sex_tot_v3 = nat_sex_v3.groupby(["preusuel", "sexe"])["nombre"].sum().reset_index()
sex_piv_v3 = sex_tot_v3.pivot(index="preusuel", columns="sexe", values="nombre").fillna(0)
dual_v3    = sex_piv_v3[(sex_piv_v3[1] >= 200) & (sex_piv_v3[2] >= 200)].index.tolist()
dual_sorted_v3 = (
    df[df.preusuel.isin(dual_v3)]
    .groupby("preusuel")["nombre"].sum()
    .sort_values(ascending=False)
    .head(50).index.tolist()
)
_pref_v3   = ["DOMINIQUE", "CAMILLE", "CLAUDE"]
default_v3 = next((n for n in _pref_v3 if n in dual_sorted_v3), dual_sorted_v3[0])
data_v3    = nat_sex_v3[nat_sex_v3.preusuel.isin(dual_sorted_v3)].copy()
name_df_v3 = pd.DataFrame({"preusuel": dual_sorted_v3})
print(f"Unisex names in dropdown: {len(dual_sorted_v3)}.  Default: {default_v3}")

# Per-name total births (both sexes), used by the mini area chart above the butterfly.
totals_v3 = (
    data_v3.groupby(["preusuel", "annais"], as_index=False)["nombre"]
    .sum()
    .rename(columns={"nombre": "year_total"})
)

# ── Params ────────────────────────────────────────────────────────────────────
name_p3 = alt.param(
    name="v3_name",
    bind=alt.binding_select(options=dual_sorted_v3, name="First name: "),
    value=default_v3,
)
ymin_p3 = alt.param(
    name="v3_ymin",
    bind=alt.binding_range(min=1900, max=2019, step=1, name="From year: "),
    value=1970,
)
ymax_p3 = alt.param(
    name="v3_ymax",
    bind=alt.binding_range(min=1901, max=2020, step=1, name="To year: "),
    value=2020,
)
step_p3 = alt.param(
    name="v3_step",
    bind=alt.binding_range(min=1, max=20, step=1, name="Step (years): "),
    value=5,
)
# Raw counts vs percent-of-name-total toggle. "percent" normalises by this
# name's own yearly total (not each sex's national total, which we tested
# and rejected — it reproduces the same invisible-bar problem at a
# different scale instead of solving it).
mode_p3 = alt.param(
    name="v3_mode",
    bind=alt.binding_radio(options=["raw", "percent"], name="Display: "),
    value="raw",
)

# ── Chart ─────────────────────────────────────────────────────────────────────
name_title_v3 = (
    alt.Chart(name_df_v3)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(300), y=alt.value(12))
    .transform_filter("datum.preusuel === v3_name")
    .properties(width=600, height=22)
)

# Mini area chart: total yearly popularity (both sexes), shares the
# butterfly's year-range filter. Answers the question the butterfly alone
# can't: is this name dying out, or just shifting gender balance?
total_line_v3 = (
    alt.Chart(totals_v3)
    .mark_area(line={"color": "#555555", "strokeWidth": 1.5}, color="#dddddd", opacity=0.6)
    .encode(
        x=alt.X(
            "annais:Q",
            title="Year",
            scale=alt.Scale(domain={"expr": "[v3_ymin, v3_ymax]"}),
            axis=alt.Axis(format="d", tickCount=8, labelAngle=0, grid=True),
        ),
        y=alt.Y("year_total:Q", title="Total births"),
        tooltip=[
            alt.Tooltip("annais:Q", title="Year", format="d"),
            alt.Tooltip("year_total:Q", title="Total births", format=","),
        ],
    )
    .transform_filter("datum.preusuel === v3_name")
    .transform_filter("datum.annais >= v3_ymin && datum.annais <= v3_ymax")
    .properties(width=600, height=70)
)

# Butterfly bars: boys extend left (val = -magnitude), girls extend right
# (val = +magnitude). magnitude is either the raw count or this sex's
# share of the name's total births that year (sums to 100% either bar).
bars_v3 = (
    alt.Chart(data_v3)
    .mark_bar()
    .encode(
        x=alt.X(
            "val:Q",
            title="← Boys | Girls →",
            axis=alt.Axis(
                labelExpr="v3_mode == 'percent' ? format(abs(datum.value), '.0f') + '%' : format(abs(datum.value), ',.0f')",
                labelAngle=0,
            ),
        ),
        y=alt.Y(
            "annais:O",
            title="Year",
            sort="ascending",
            axis=alt.Axis(labelAngle=0),
        ),
        color=alt.Color(
            "gender:N",
            scale=alt.Scale(domain=["Boys", "Girls"], range=["#4e78a8", "#e7729a"]),
            legend=alt.Legend(title="Sex"),
        ),
        tooltip=[
            alt.Tooltip("annais:O",  title="Year"),
            alt.Tooltip("gender:N",  title="Sex"),
            alt.Tooltip("nombre:Q",  title="Births", format=","),
            alt.Tooltip("pct:Q",     title="% of that year's births", format=".1f"),
        ],
    )
    .transform_filter("datum.preusuel === v3_name")
    .transform_filter("datum.annais >= v3_ymin && datum.annais <= v3_ymax")
    .transform_filter("(datum.annais - v3_ymin) % v3_step === 0")
    .transform_joinaggregate(year_total="sum(nombre)", groupby=["annais"])
    .transform_calculate("pct", "datum.year_total > 0 ? datum.nombre / datum.year_total * 100 : 0")
    .transform_calculate("magnitude", "v3_mode == 'percent' ? datum.pct : datum.nombre")
    .transform_calculate("val",    "datum.sexe === 1 ? -datum.magnitude : datum.magnitude")
    .transform_calculate("gender", "datum.sexe === 1 ? 'Boys' : 'Girls'")
    .properties(width=600, height=400)
)

centre_v3 = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="#333333", strokeWidth=1.5)
    .encode(x=alt.X("x:Q"))
)

v3 = (
    alt.vconcat(
        name_title_v3,
        total_line_v3,
        alt.layer(bars_v3, centre_v3),
        spacing=4,
    )
    .add_params(name_p3, ymin_p3, ymax_p3, step_p3, mode_p3)
    .properties(
        title=alt.TitleParams(
            "Gender butterfly chart - Boys vs Girls",
            subtitle="Top-50 unisex names by total popularity (1900-2020). Toggle raw counts / percent-of-year-total to read lopsided names; the area chart above tracks the name's overall popularity.",
            subtitleFontSize=11,
            subtitleColor="gray",
        )
    )
    .configure_view(strokeWidth=0)
)

v3
